In [1]:
import pandas as pd
import sys
from pathlib import Path

from sklearn.model_selection import StratifiedGroupKFold

sys.path.append(str(Path("../src").resolve()))

from exoplanet_detector.features.feature_selection import (
    FINAL_FEATURE_COLUMNS,
    RIGHT_SKEWED_COLUMNS,
    LEFT_SKEWED_COLUMNS,
    PHYSICAL_INTERVALS,
)

from scipy.stats import loguniform, randint
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from exoplanet_detector.features.pipelines import build_preprocessing_pipeline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)


In [2]:
KOI_train_set = pd.read_csv("../data/processed/KOI_train_set.csv")
KOI_test_set = pd.read_csv("../data/processed/KOI_test_set.csv")
K2P_set = pd.read_csv("../data/processed/K2P_set.csv")

The full preprocessing pipeline was implemented and imported from `pipelines.py`. Now, 5 different models will be trained and tuned and the one performing the best will be chosen.

In [3]:
X_train = KOI_train_set.drop(columns=["label", "group_id"])
y_train = KOI_train_set["label"]
group_id = KOI_train_set["group_id"]

In [4]:
X_test = KOI_test_set.drop(columns=["label", "group_id"])
y_test = KOI_test_set["label"]

In [5]:
X_k2p = K2P_set.drop(columns=["label", "group_id"])
y_k2p = K2P_set["label"]

In [6]:
model_specs = {
    "logreg": {
        "estimator": LogisticRegression(max_iter=5000, class_weight="balanced"),
        "with_scaling": True,
        "params": {"clf__C": loguniform(1e-3, 1e2)},
    },
    "svc_rbf": {
        "estimator": SVC(probability=True, class_weight="balanced"),
        "with_scaling": True,
        "params": {"clf__C": loguniform(1e-2, 1e2), "clf__gamma": loguniform(1e-4, 1e-1)},
    },
    "knn": {
        "estimator": KNeighborsClassifier(),
        "with_scaling": True,
        "params": {"clf__n_neighbors": randint(3, 41), "clf__weights": ["uniform", "distance"]},
    },
    "rf": {
        "estimator": RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
        "with_scaling": False,
        "params": {"clf__n_estimators": randint(200, 800), "clf__max_depth": [None, 5, 10, 20]},
    },
    "hgb": {
        "estimator": HistGradientBoostingClassifier(random_state=42),
        "with_scaling": False,
        "params": {"clf__learning_rate": loguniform(1e-3, 1e-1), "clf__max_leaf_nodes": randint(15, 63)},
    },
}

In [7]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
search_results = {}

In [8]:
for name, spec in model_specs.items():
    pipe = Pipeline([
        ("preprocess", build_preprocessing_pipeline(with_scaling=spec["with_scaling"])),
        ("clf", spec["estimator"]),
    ])

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=spec["params"],
        n_iter=25,
        scoring="average_precision",
        cv=cv,
        n_jobs=-1,
        random_state=42,
        refit=True,
        verbose=1,
    )

    search.fit(X_train, y_train, groups=group_id)
    search_results[name] = search